In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. 生成交易日日期（2020-01-02至2021-10-31，剔除周末和主要节假日）
start_date = datetime(2020, 1, 2)
end_date = datetime(2021, 10, 31)
all_dates = pd.date_range(start=start_date, end=end_date)

# 过滤周末（只保留周一至周五）
trade_dates = [date for date in all_dates if date.weekday() < 5]  # 0=周一, 4=周五

# 手动剔除主要节假日（2020-2021年A股休市日期）
holidays = [
    # 2020年节假日
    "2020-01-24", "2020-01-25", "2020-01-26", "2020-01-27", "2020-01-28",  # 春节
    "2020-04-04", "2020-04-05", "2020-04-06",  # 清明
    "2020-05-01", "2020-05-02", "2020-05-03", "2020-05-04", "2020-05-05",  # 五一
    "2020-06-25", "2020-06-26", "2020-06-27",  # 端午
    "2020-10-01", "2020-10-02", "2020-10-03", "2020-10-04", "2020-10-05", "2020-10-06", "2020-10-07",  # 国庆
    # 2021年节假日
    "2021-02-11", "2021-02-12", "2021-02-13", "2021-02-14", "2021-02-15", "2021-02-16", "2021-02-17",  # 春节
    "2021-04-03", "2021-04-04", "2021-04-05",  # 清明
    "2021-05-01", "2021-05-02", "2021-05-03", "2021-05-04",  # 五一
    "2021-06-12", "2021-06-13", "2021-06-14",  # 端午
    "2021-10-01", "2021-10-02", "2021-10-03", "2021-10-04", "2021-10-05", "2021-10-06", "2021-10-07"  # 国庆
]
holidays = [datetime.strptime(day, "%Y-%m-%d") for day in holidays]

# 最终交易日（排除节假日）
final_dates = [date for date in trade_dates if date not in holidays]
# 确保数据量约400条（截取前400条）
final_dates = final_dates[:400]
date_strs = [date.strftime("%Y-%m-%d") for date in final_dates]


# 2. 生成模拟收盘价（带趋势和波动）
np.random.seed(42)  # 固定随机种子，确保结果可复现
initial_price = 28.5  # 初始价格（与之前数据衔接）
prices = [initial_price]

# 分阶段设定趋势（模拟真实股票走势）
# 阶段1：2020-01至2020-06，缓慢上涨（趋势+0.02/天）
# 阶段2：2020-07至2020-12，震荡整理（趋势±0.01）
# 阶段3：2021-01至2021-06，快速上涨（趋势+0.05/天）
# 阶段4：2021-07至2021-10，小幅下跌（趋势-0.03/天）
stage1_end = 120  # 前120天（约6个月）
stage2_end = 240  # 121-240天（接下来6个月）
stage3_end = 360  # 241-360天（接下来6个月）
stage4_end = 400  # 361-400天（最后40天）

for i in range(1, 400):
    if i <= stage1_end:
        trend = 0.02  # 阶段1：上涨趋势
    elif i <= stage2_end:
        trend = np.random.choice([-0.01, 0.01])  # 阶段2：震荡
    elif i <= stage3_end:
        trend = 0.05  # 阶段3：快速上涨
    else:
        trend = -0.03  # 阶段4：小幅下跌
    
    # 随机波动（±2%以内）
    波动 = np.random.normal(loc=trend, scale=0.6)  # 标准差0.6控制波动幅度
    next_price = prices[-1] + 波动
    # 确保价格为正数（实际股票价格不会为负）
    next_price = max(5, round(next_price, 2))  # 最低5元，保留2位小数
    prices.append(next_price)


# 3. 生成CSV文件
df = pd.DataFrame({
    "date": date_strs,
    "close_price": prices
})

# 保存为CSV（当前目录）
df.to_csv("pingan_train_400.csv", index=False, encoding="utf-8")

print(f"CSV文件生成成功！共{len(df)}条数据，文件名为：pingan_stock_400.csv")

CSV文件生成成功！共400条数据，文件名为：pingan_stock_400.csv


In [4]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. 生成测试集日期（确保与价格长度一致）
# 训练集最后日期（根据实际训练集最后一条日期调整，这里假设为2021-10-29）
train_last_date = datetime(2021, 10, 29)
start_date = train_last_date + timedelta(days=1)
end_date = datetime(2022, 6, 30)  # 延长结束日期，确保有足够交易日

# 生成所有候选日期并过滤周末
all_dates = pd.date_range(start=start_date, end=end_date)
test_trade_dates = [date for date in all_dates if date.weekday() < 5]  # 仅保留周一至周五

# 剔除2021-11至2022-06的节假日（A股休市日期）
test_holidays = [
    "2021-10-30", "2021-10-31",  # 补全10月非交易日
    "2021-12-31",  # 元旦前调休
    "2022-01-01", "2022-01-02", "2022-01-03",  # 元旦
    "2022-01-31", "2022-02-01", "2022-02-02", "2022-02-03", "2022-02-04", "2022-02-05", "2022-02-06",  # 春节
    "2022-04-02", "2022-04-03", "2022-04-04",  # 清明
    "2022-04-30", "2022-05-01", "2022-05-02", "2022-05-03", "2022-05-04",  # 五一
    "2022-06-03", "2022-06-04", "2022-06-05"  # 端午
]
test_holidays = [datetime.strptime(day, "%Y-%m-%d") for day in test_holidays]

# 过滤节假日后，取前150条交易日（若不足150会自动取实际数量，避免长度不匹配）
final_test_dates = [date for date in test_trade_dates if date not in test_holidays][:150]
test_date_strs = [date.strftime("%Y-%m-%d") for date in final_test_dates]
n_samples = len(test_date_strs)  # 实际样本数（确保与价格长度一致）


# 2. 生成测试集收盘价（长度与日期严格一致）
np.random.seed(43)
last_train_price = 52.3  # 训练集最后一条价格（根据实际值调整）
test_prices = [last_train_price]

# 分阶段趋势（根据实际样本数调整阶段）
stage1_end = int(n_samples * 1/3)  # 前1/3
stage2_end = int(n_samples * 2/3)  # 中间1/3
stage3_end = n_samples  # 最后1/3

for i in range(1, n_samples):  # 循环次数与日期长度一致
    if i <= stage1_end:
        trend = -0.08  # 加速下跌
    elif i <= stage2_end:
        trend = 0.06  # 触底反弹
    else:
        trend = np.random.choice([-0.02, 0.02])  # 震荡整理
    
    # 随机波动（标准差0.8）
    fluctuation = np.random.normal(loc=trend, scale=0.8)  # 修正变量名为英文，避免语法错误
    next_price = test_prices[-1] + fluctuation
    next_price = max(8, round(next_price, 2))  # 确保价格为正数
    test_prices.append(next_price)


# 3. 生成CSV（此时日期和价格长度一定一致）
test_df = pd.DataFrame({
    "date": test_date_strs,
    "close_price": test_prices
})

test_df.to_csv("pingan_test_150.csv", index=False, encoding="utf-8")
print(f"测试集生成成功！共{len(test_df)}条数据（日期和价格长度一致）")

测试集生成成功！共150条数据（日期和价格长度一致）


In [17]:
import csv
import requests
from datetime import datetime
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def get_price_history(market_hash_name):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Cookie": "timezoneOffset=28800,0; wants_mature_content_apps=1788780; recentlyVisitedAppHubs=1788780%2C359550; sessionid=a36d9a212bc1fa56b271db02; steamCountry=CN%7C5b2cba9246d1d5cf4d8178bbbc9553ce; steamLoginSecure=76561199170673689%7C%7CeyAidHlwIjogIkpXVCIsICJhbGciOiAiRWREU0EiIH0.eyAiaXNzIjogInI6MDAwRF8yNzJDMzc2RV82NzM3MCIsICJzdWIiOiAiNzY1NjExOTkxNzA2NzM2ODkiLCAiYXVkIjogWyAid2ViOmNvbW11bml0eSIgXSwgImV4cCI6IDE3NjIyMjg2NDYsICJuYmYiOiAxNzUzNTAwNjcwLCAiaWF0IjogMTc2MjE0MDY3MCwgImp0aSI6ICIwMDBDXzI3MkMzNzZFXzE1MkE2IiwgIm9hdCI6IDE3NjIxNDA2NzAsICJydF9leHAiOiAxNzgwNDkyMTMwLCAicGVyIjogMCwgImlwX3N1YmplY3QiOiAiMTEyLjU0LjEzLjE3NiIsICJpcF9jb25maXJtZXIiOiAiMTEyLjU0LjEzLjE3NiIgfQ.F5awC2bJZahTGn8ME74n3RCqMs-x-DMJGcMP-D1jwIOnzNnCvu9yaIrkiraSHz0MfAxGtNQKASx64Wvq0MsqAw; browserid=124538120351160203; webTradeEligibility=%7B%22allowed%22%3A1%2C%22allowed_at_time%22%3A0%2C%22steamguard_required_days%22%3A15%2C%22new_device_cooldown_days%22%3A0%2C%22time_checked%22%3A1762140826%7D; timezoneName=America%2FLos_Angeles"  # 保持之前的Cookie不变
    }
    url = f"https://steamcommunity.com/market/pricehistory/?appid=730&market_hash_name={market_hash_name}"
    
    # 配置代理（替换为你的代理端口，示例为7890）
    proxies = {
        "https": "http://127.0.0.1:7890",
        "http": "http://127.0.0.1:7890"
    }
    
    retry_strategy = Retry(total=3, backoff_factor=1)
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session = requests.Session()
    session.mount("https://", adapter)
    
    try:
        # 添加proxies参数
        response = session.get(url, headers=headers, proxies=proxies, timeout=15)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"请求失败，状态码：{response.status_code}")
            return None
    except Exception as e:
        print(f"请求出错：{str(e)}")
        return None

# 后续save_to_csv和调用逻辑保持不变...
def save_to_csv(data, filename):
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Date', 'Median Price'])
        recent_prices = data['prices'][-30:] if len(data['prices']) > 30 else data['prices']
        for entry in recent_prices:
            date_str = datetime.utcfromtimestamp(entry[0]).strftime('%Y-%m-%d')
            price = entry[1]
            writer.writerow([date_str, price])

market_hash_name = "AK-47%20%7C%20Redline%20%28Field-Tested%29"
data = get_price_history(market_hash_name)

if data and 'prices' in data:
    save_to_csv(data, 'ak_redline_recent_prices.csv')
    print("最近一个月数据已保存到 'ak_redline_recent_prices.csv'")
else:
    print("未获取到有效数据，请检查代理端口或网络连接")

请求出错：HTTPSConnectionPool(host='steamcommunity.com', port=443): Max retries exceeded with url: /market/pricehistory/?appid=730&market_hash_name=AK-47%20%7C%20Redline%20%28Field-Tested%29 (Caused by ProxyError('Unable to connect to proxy', NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x774d59e10fa0>: Failed to establish a new connection: [Errno 111] Connection refused')))
未获取到有效数据，请检查代理端口或网络连接
